# **Theoretical Questions**:

**Question 1 :**

What is a Calculated Field and why is it important in business dashboards?

**ANS.**


A Calculated Field is a user-defined custom field created within a Business Intelligence (BI) tool (such as Tableau, Power BI, or Looker) by applying mathematical formulas, logical statements, or text manipulations to existing data columns.

Instead of changing or rewriting the raw database table itself, a calculated field acts as a virtual column. It processes data on the fly (dynamically) when the dashboard runs. For example, if your dataset contains columns for Sales and Cost, you can write a calculated field formula (Sales - Cost) to compute a brand-new column called Profit.

**Why is it Important in Business Dashboards?**

Calculated Fields are the backbone of analytical dashboards, turning raw data rows into meaningful business intelligence. Here is why they are indispensable:

**1. Enables Hidden Insights & New Metrics**

Raw data rarely contains every metric a business needs. Calculated fields allow users to build key performance indicators (KPIs) that didn't exist originally.

Example: Turning absolute numbers into ratios like Profit Margin % or Customer Retention Rate.

**2. Saves Time and Costs (No ETL/Database Dependency)**
Without calculated fields, a business analyst would have to ask a database administrator (DBA) or data engineer to modify the underlying data tables or write complex SQL ETL (Extract, Transform, Load) pipelines. This process can take days or weeks. Calculated fields allow dashboard creators to build and test metrics instantly.

**3. Implements Custom Business Logic**

Every organization has unique business rules that standard data doesn't account for. Calculated fields allow dashboards to apply custom rules using logical statements (like IF-THEN-ELSE).

Example: As seen in your Blinkit task, classifying orders dynamically into "Fast" or "Delayed" based on whether delivery time is under or over 30 minutes.

**4. Ensures Dynamic, Real-Time Interactivity**
Calculated fields adapt automatically when filters are adjusted. If a business user filters a dashboard to look only at "Mumbai" or the "Dairy Category," the calculated field recalculates the numbers instantly based solely on the active selection.

**5. Simplifies Complex Data for Executives**

Data is often stored in highly technical or fragmented formats (e.g., separate columns for First_Name and Last_Name, or cryptic codes like 1 and 0 for active/inactive users). Calculated fields help clean and merge fields to make dashboards intuitive for decision-makers.

Example: Changing binary codes into human-readable text labels (IF [Status] = 1 THEN "Active" ELSE "Churned" END).

**Question 2 :**

 Why can’t parameters change visuals directly in Tableau?

In Tableau, parameters cannot change visuals directly because they are fundamentally passive, static containers of data. They do not possess any native "awareness" of your data source, fields, or worksheets.

Here is a detailed breakdown of why this architectural limitation exists and how it works:

1. Parameters Are "Global Containers," Not OperationsA parameter is simply a user-controlled variable (like a text box, slider, or dropdown menu) that holds a single value at a time (e.g., the number 5, the date 2026-06-18, or the text "Mumbai").Think of a parameter like a thermostat dial.

 Turning the dial to 72 degrees doesn't change the temperature of the room by itself; it requires a wire connecting it to the heating/cooling system to actually trigger a change.

2. They Lack Context and Data ConnectionsStandard fields in Tableau (like Sales or City) are hardwired directly to your data source. When you drag City to the Columns shelf, Tableau automatically queries the database and draws bars for every city.

A parameter, however, sits completely independent of your dataset. Because it does not map to rows or columns on its own, dragging a raw parameter onto a visual workspace does nothing more than display its current fixed value over and over again.

3. How the "Bridge" Works (Calculated Fields & Filters)To make a parameter actually change a visual, you must build a "bridge" that connects that parameter to your data. This bridge is almost always a Calculated Field or a Filter.

For Filters (e.g., Top N Analysis): If you want a parameter to dynamically filter the chart to show the top $N$ products, the parameter just holds the number (e.g., 10). You must then go to the Product Name filter settings and tell Tableau: "Filter by Top [Your Parameter Name] items.

"For Dynamic Dimensions/Metrics: If you want a dropdown parameter to change a chart from showing Sales to showing Profit, you must write a calculated field that interprets the parameter value:


```
CASE [Choose Metric Parameter]
  WHEN "Sales" THEN [Sales]
  WHEN "Profit" THEN [Profit]
END
```



**Question 3 :**

When should EXCLUDE LOD be used?

**ANS.**

An EXCLUDE Level of Detail (LOD) expression is used in Tableau when you want to calculate a metric at a higher, coarser level of aggregation than what is currently displayed in your visualization layout.

Essentially, it tells Tableau: "Look at the dimensions I have placed in this specific chart, but explicitly ignore (exclude) certain ones when computing this calculation."



```
{ EXCLUDE [Dimension to Ignore] : SUM([Sales]) }
```



Here are the primary business scenarios and use cases for when an EXCLUDE LOD should be used:

1. Calculating Proportions and Percent of Total
When you need to know how much a granular component contributes to a larger category bucket, you need both the local row value and the broader category's total available in the same view.

Scenario: You have a bar chart showing sales broken down by Category and Product Name. You want to see what percentage of the Category's overall sales each individual Product represents.

How EXCLUDE helps: You can exclude the [Product Name] dimension to calculate the total sales for the overall Category, and then divide the individual product sales by this number.

Formula: ```tableau
SUM([Sales]) / SUM({ EXCLUDE [Product Name] : SUM([Sales]) })


2. Side-by-Side Benchmarking (Comparing Rows to Group Averages)
When you want to compare individual items directly against the average or total of the entire group they belong to.

Scenario: You are analyzing regional performance. Your view displays individual [City] rows inside a specific [Region]. You want a column that shows the average sales of the overall Region right next to each City so you can instantly identify underperforming territories.

How EXCLUDE helps: By excluding [City], the calculation rolls back up to the Region level, creating a static benchmark column across all cities within that region.

Formula: ```tableau
{ EXCLUDE [City] : AVG([Sales]) }


3. Avoiding the Interactivity Limits of FIXED LODs
While a FIXED LOD can also calculate numbers at higher levels, it completely ignores standard dashboard filters unless those filters are manually added to "Context."

When to choose EXCLUDE over FIXED: If you want your benchmark calculation to remain dynamic and automatically change whenever a user clicks on standard dashboard filters (like Date, Segment, or Payment Mode), you should use EXCLUDE. Because EXCLUDE is evaluated after standard Dimension filters in Tableau's Order of Operations, it respects user selections flawlessly.

4. Overcoming "Asymmetrical" Chart Layouts
Sometimes business users want to display a highly granular dimension in a chart but want a reference line or color gradient based entirely on a broader dimension.

Scenario: A scatterplot charting individual [Customer Name] points, but you want to color-code those customers based on whether their purchase behavior is above or below their overall [City] average.

How EXCLUDE helps: Exclude [Customer Name] from the calculation to establish the baseline city average for that specific point.

**Question 4 :**

Difference between FIXED and INCLUDE LOD expressions?

**ANS**.

The primary difference between FIXED and INCLUDE Level of Detail (LOD) expressions in Tableau lies in how they interact with the dimensions currently placed on your visualization shelves and how they are affected by filters.


**FIXED LOD**


A FIXED expression locks the calculation to the specific dimension(s) you name in the formula, regardless of how deep or wide you slice your visualization.

Syntax: { FIXED [City] : SUM([Sales]) }

How it works: Tableau creates a separate, isolated sub-query that sums up sales grouped only by City. Even if you drag Category onto your Rows shelf to break down your chart, the FIXED calculation will keep printing the absolute total sales for the entire city on every row.

Filter Behavior: If a user filters out a specific category (e.g., hiding "Snacks"), a standard FIXED LOD will not change its value because it executes before that filter is processed.

**INCLUDE LOD**

An INCLUDE expression tells Tableau to calculate the metric at a more detailed (lower) level than what is visible, but still combine it with the structure of your visualization.

Syntax: { INCLUDE [Product Name] : AVG([Sales]) }

How it works: If your chart only shows a high-level summary like Category, the INCLUDE expression tells Tableau: "Look down at the hidden product level first, find the average sales for each product, and then roll those results back up to the visible Category level."

Filter Behavior: Because INCLUDE executes after standard dimension filters, if you filter out a specific payment mode or date range, the calculation dynamically adjusts immediately

# **Practical Questions** :

**Dataset:**
https://docs.google.com/spreadsheets/d/1hnPNN1bYDIn4NQjGAjcpfIqdYBdtDaumgH7gmaoTpfw/edit?usp=sharing

**Task 1: Delivery Performance Classification (Calculated Field)**

**Business Need**

Operations wants to track delivery efficiency.

**Task**

Classify orders as Fast or Delayed.

**Condition**

* Delivery ≤ 30 min → Fast

* Else → Delayed

Task 1: Delivery Performance Classification (Calculated Field)
Business Need: Operations wants to track delivery efficiency to understand what proportion of orders are meeting service level agreements (SLAs).

Tableau Calculated Field Formula:



```
IF [Delivery Time Min] <= 30 THEN "Fast"
ELSE "Delayed"
END
```

Power BI DAX Formula:

```
Delivery Performance = IF('Blinkit Orders'[Delivery_Time_Min] <= 30, "Fast", "Delayed")
```

Detailed Insights from Data:

Total Orders analyzed: $1,000$

Delayed Deliveries ($> 30\text{ min}$): $567$ orders ($56.7\%$)

Fast Deliveries ($\le 30\text{ min}$): $433$ orders ($43.3\%$)

**Task 2: Loss-Making Orders Identification**


**Business Need**

High sales do not always mean high profit.

**Task**

Identify orders resulting in loss.

Business Need: High sales values can mask unprofitable transactions. Identifying individual orders resulting in a loss is vital to safeguarding profit margins.

Tableau Calculated Field Formula:



```
IF [Profit] < 0 THEN "Loss-Making"
ELSE "Profitable"
END
```
Power BI DAX Formula


```
Is Loss Making = IF('Blinkit Orders'[Profit] < 0, "Loss-Making", "Profitable")
```


Detailed Insights from Data:

Number of Loss-Making Orders: $0$ out of $1,000$ orders.

Profit Range: The minimum profit for a single order in this dataset is $\$3.97$, and the maximum profit is $\$367.57$. The average profit per order stands at $\$93.06$.


**Task 3: Top N Products Analysis (Parameter)**

**Business Need**

The product team wants to focus on top-performing products.

**Task**

Create a Top N Products dynamic analysis.

Top N Products Analysis (Parameter)

Business Need: The product team requires a dynamic way to adjust rankings and focus on top-performing items without rebuilding charts.

Tableau Implementation

Steps:Create a Parameter: Name it Top N Products. Set the Data Type to Integer and select Range or List (e.g., values $3, 5, 10$).

Create a Filter Set: Right-click the Product Name dimension $\rightarrow$ Create $\rightarrow$ Set.

Go to the Top tab, choose By field, select Top, and select your Top N Products parameter from the dropdown. Set it to aggregate by Sales (Sum) or Profit (Sum).

Apply Filter: Drag this newly created set to the Filters Shelf.

**Task 4: City-Level Benchmarking (FIXED LOD)**

**Business Need**

Compare city performance independent of category filters.

**Task**

Calculate average sales per city using FIXED LOD

Business Need: Compare city performance independently of category filters applied to the dashboard view.

Tableau FIXED LOD Formula:



```
{ FIXED [City] : AVG([Sales]) }
```



**Task 5: Customer/Product Detail Impact (INCLUDE LOD)**

**Business Need**

The sales team wants average sales considering product-level detail, even when product is not shown.

**Task**

Create a calculation that finds the average sales at the product level, but allows the result to be used at a
higher level (such as Category or Region). Use INCLUDE LOD.

Business Need: Calculate average sales taking product-level insights into account, even when the individual products are hidden and the view is aggregated at a higher level (like Category).

Tableau INCLUDE LOD Formula:



```
{ INCLUDE [Product Name] : AVG([Sales]) }
```

